### Loading the Dataset
* **Problem:** Need to load our raw e-commerce data into Python to inspect its structure[cite: 3, 7].
* **What I Did:** Imported `pandas` and `numpy`, loaded the CSV file from the `dataset` folder, and checked the shape and first few rows[cite: 7].

In [2]:
import pandas as pd
import numpy as np

# Load dataset from subfolder
df = pd.read_csv("dataset/ecommerce_sales_customer_analytics_150k.csv")

# Display shape (rows, columns) and first 5 rows
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (138116, 46)


,order_id,order_date,order_time,order_status,sales_channel,customer_id,customer_name,customer_age,gender,customer_segment,...,discount_amount,tax_amount,shipping_cost,net_sales,product_cost,profit,profit_margin_percentage,customer_lifetime_value,is_repeat_customer,customer_order_count
0,ORD-301242,2023-11-06,16:37:47,Completed,Mobile App,CUST-003102,Jasmine Ryan,55,Male,Consumer,...,477.89,61.07,12.03,945.40,588.33,345.04,36.50,12459.68,True,11
1,ORD-773460,2025-12-24,01:22:36,Completed,Website,CUST-003124,Scott Chase,26,Male,Premium,...,1456.06,320.83,9.00,2018.41,2031.88,-22.47,-1.11,13032.48,True,11
2,ORD-449374,2021-07-05,14:24:21,Completed,Mobile App,CUST-012496,Marc Wheeler,69,Female,Premium,...,97.30,29.79,13.05,468.35,257.73,197.57,42.18,6159.44,True,6
3,ORD-567636,2023-01-21,07:20:26,Completed,Social Media,CUST-023928,Jennifer Smith,65,Male,Consumer,...,53.34,34.39,20.85,546.52,277.84,247.83,45.35,5638.30,True,7
4,ORD-820028,2022-05-13,09:46:21,Completed,Social Media,CUST-012730,Jessica Wang,34,Female,Consumer,...,133.63,421.35,23.03,2785.27,1231.88,1530.36,54.94,13240.10,True,8


### Checking Data Types & Missing Values
* **Problem:** Some columns might have wrong data types or missing entries[cite: 3, 7].
* **What I Did:** Ran `df.info()` to check column types and listed all columns that contained null values to see where
* clean-up was needed[cite: 3, 7].

In [2]:
# Check data types and overall null summary
df.info()

# View exact missing value counts for columns that have nulls
missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0]
print("\nColumns with Missing Values:\n", missing_data)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 138116 entries, 0 to 138115
Data columns (total 46 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   order_id                  138116 non-null  object 
 1   order_date                138116 non-null  object 
 2   order_time                138116 non-null  object 
 3   order_status              138116 non-null  object 
 4   sales_channel             138116 non-null  object 
 5   customer_id               138116 non-null  object 
 6   customer_name             138116 non-null  object 
 7   customer_age              138116 non-null  int64  
 8   gender                    138116 non-null  object 
 9   customer_segment          138116 non-null  object 
 10  customer_type             138116 non-null  object 
 11  customer_city             138116 non-null  object 
 12  customer_state            138116 non-null  object 
 13  customer_country          138116 non-null  o

### Auditing Data Quality & Checking Calculations
* **Problem:** Need to spot formatting errors, weird number ranges, or broken sales formulas before fixing
* anything[cite: 3, 7].
* **What I Did:** Checked unique values in text columns, looked at summary stats (`min`, `max`, `mean`), and verified
* if `net_sales` matched `gross_sales - discount_amount`[cite: 3, 7].

In [3]:
# --- 1. Unique Values & Data Formatting Check ---
print("=== 1. CATEGORICAL VALUES & TEXT FORMATTING ===")
categorical_cols = ['sales_channel', 'order_status', 'payment_method', 'shipping_method', 'customer_segment', 'gender']
for col in categorical_cols:
    if col in df.columns:
        print(f"\n{col} ({df[col].nunique()} unique values):")
        print(df[col].value_counts().head(5))

# --- 2. Outliers & Numeric Extremes ---
print("\n" + "="*50)
print("=== 2. NUMERIC SUMMARY & OUTLIER CHECK ===")
numeric_cols = ['quantity', 'gross_sales', 'discount_amount', 'shipping_cost', 'net_sales', 'profit', 'profit_margin_percentage']
print(df[numeric_cols].describe().T[['min', 'mean', 'max', 'std']])

# --- 3. Logical Integrity Checks ---
print("\n" + "="*50)
print("=== 3. LOGICAL INTEGRITY CHECKS ===")
# Check if net_sales == gross_sales - discount_amount
net_sales_check = (df['net_sales'] - (df['gross_sales'] - df['discount_amount'])).abs() > 0.01
print("Net Sales Miscalculations:", net_sales_check.sum())

# Check for negative quantities or sales
print("Negative Quantities:", (df['quantity'] <= 0).sum())
print("Negative Gross Sales:", (df['gross_sales'] < 0).sum())

# Check date ranges
df['order_date_dt'] = pd.to_datetime(df['order_date'], errors='coerce')
print("Invalid Date Strings:", df['order_date_dt'].isnull().sum())
print("Date Range:", df['order_date_dt'].min(), "to", df['order_date_dt'].max())

=== 1. CATEGORICAL VALUES & TEXT FORMATTING ===

sales_channel (4 unique values):
sales_channel
Mobile App      55318
Website         48237
Marketplace     20698
Social Media    13863
Name: count, dtype: int64

order_status (4 unique values):
order_status
Completed    113559
Returned       9462
Cancelled      8398
Pending        6697
Name: count, dtype: int64

payment_method (7 unique values):
payment_method
Credit Card         41460
Debit Card          27635
PayPal              20750
Digital Wallet      16382
Cash on Delivery    13863
Name: count, dtype: int64

shipping_method (4 unique values):
shipping_method
Standard    62006
Express     34373
Economy     27824
Same Day    13913
Name: count, dtype: int64

customer_segment (4 unique values):
customer_segment
Consumer    75636
Premium     34746
VIP         13957
Business    13777
Name: count, dtype: int64

gender (3 unique values):
gender
Female        66439
Male          66284
Non-Binary     5393
Name: count, dtype: int64

=== 2. NU

### Step 1: Recalculating Financial Metrics
* **Problem:** `net_sales`, `profit`, and `profit_margin_percentage` had math errors in the original file[cite: 3, 7].
* **What I Did:** Recalculated all three columns using standard formulas rounded to 2 decimal places:
  * `net_sales = gross_sales - discount_amount`[cite: 3, 7]
  * `profit = net_sales - product_cost`[cite: 3, 7]
  * `profit_margin_percentage = (profit / net_sales) * 100`[cite: 3, 7]

In [5]:
# --- STEP 1: Standardize Financial Calculations ---

# 1. Standardize net_sales: gross_sales minus discount_amount
df['net_sales'] = (df['gross_sales'] - df['discount_amount']).round(2)

# 2. Re-calculate profit: net_sales minus product_cost
df['profit'] = (df['net_sales'] - df['product_cost']).round(2)

# 3. Re-calculate profit_margin_percentage: (profit / net_sales) * 100
df['profit_margin_percentage'] = ((df['profit'] / df['net_sales']) * 100).round(2)

# 4. Verify the math recalculation on a sample of rows
print("=== Step 1 Complete: Sample Financial Recalculation ===")
df[['gross_sales', 'discount_amount', 'net_sales', 'product_cost', 'profit', 'profit_margin_percentage']].head()

=== Step 1 Complete: Sample Financial Recalculation ===


,gross_sales,discount_amount,net_sales,product_cost,profit,profit_margin_percentage
0,1350.19,477.89,872.30,588.33,283.97,32.55
1,3144.64,1456.06,1688.58,2031.88,-343.30,-20.33
2,522.81,97.30,425.51,257.73,167.78,39.43
3,544.62,53.34,491.28,277.84,213.44,43.45
4,2474.52,133.63,2340.89,1231.88,1109.01,47.38


### Step 2: Filling Missing Values
* **Problem:** Over 200,000 missing cells across 9 columns (returns, marketing, reviews, and delivery times)[cite: 3, 7].
* **What I Did:** Filled missing text cells with clear labels (`'Not Returned'`, `'None'`, `'Organic/Direct'`, `'No Coupon'`, `'No Feedback'`, `'No Review'`) and replaced missing numbers with the median value for that column[cite: 3, 7].

In [4]:
# --- STEP 2: Impute Missing Values Based on Domain Rules ---

# 1. Fill categorical missing values with clear business labels
df['return_status'] = df['return_status'].fillna('Not Returned')
df['return_reason'] = df['return_reason'].fillna('None')
df['campaign_name'] = df['campaign_name'].fillna('Organic/Direct')
df['coupon_code'] = df['coupon_code'].fillna('No Coupon')
df['review_sentiment'] = df['review_sentiment'].fillna('No Feedback')
df['customer_review'] = df['customer_review'].fillna('No Review')

# 2. Impute numerical missing values using column medians
df['delivery_days'] = df['delivery_days'].fillna(df['delivery_days'].median())
df['estimated_delivery_days'] = df['estimated_delivery_days'].fillna(df['estimated_delivery_days'].median())
df['customer_rating'] = df['customer_rating'].fillna(df['customer_rating'].median())

# 3. Verify that all missing values are handled
remaining_nulls = df.isnull().sum().sum()
print("=== Step 2 Complete ===")
print("Total Remaining Missing Values:", remaining_nulls)

=== Step 2 Complete ===
Total Remaining Missing Values: 0


### Step 3: Cleaning Text & Formatting Dates
* **Problem:** `order_date` was saved as plain text, and categorical columns had inconsistent spacing or capitalization[cite: 3, 7].
* **What I Did:** Converted `order_date` to `datetime64` and cleaned up key text columns (`sales_channel`, `order_status`, `payment_method`, etc.) using `.strip()` and `.title()`[cite: 3, 7].

In [5]:
# --- STEP 3: Text Standardization & Datetime Parsing ---

# 1. Convert order_date to datetime data type
df['order_date'] = pd.to_datetime(df['order_date'])

# 2. Strip hidden spaces and apply Title Case formatting to key categorical text columns
text_cols = ['sales_channel', 'order_status', 'payment_method', 'shipping_method', 'customer_segment', 'gender']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

# 3. Verify data types for order_date and text fields
print("=== Step 3 Complete ===")
print("order_date data type:", df['order_date'].dtype)
print("\nSample formatted text columns:")
df[text_cols].head(3)

=== Step 3 Complete ===
order_date data type: datetime64[ns]

Sample formatted text columns:


,sales_channel,order_status,payment_method,shipping_method,customer_segment,gender
0,Mobile App,Completed,Digital Wallet,Standard,Consumer,Male
1,Website,Completed,Debit Card,Economy,Premium,Male
2,Mobile App,Completed,Cash On Delivery,Express,Premium,Female


### Step 4: Checking Duplicates & Exporting Clean CSV
* **Problem:** Need to make sure there are no identical duplicate rows and save our cleaned dataset[cite: 3, 7].
* **What I Did:** Verified 0 duplicate rows existed and exported the final dataframe to `dataset/cleaned_ecommerce_sales_150k.csv`[cite: 3, 7].

In [6]:
# --- STEP 4: Duplicate Check & Save Cleaned Dataset ---

# 1. Check for identical row duplicates
duplicate_count = df.duplicated().sum()
print("Number of duplicate rows found:", duplicate_count)

# 2. Drop duplicates if any exist
if duplicate_count > 0:
    df = df.drop_duplicates()
    print("Duplicates removed. New dataset shape:", df.shape)

# 3. Export clean dataset to CSV
output_path = "dataset/cleaned_ecommerce_sales_150k.csv"
df.to_csv(output_path, index=False)

print("=== Step 4 Complete ===")
print(f"Cleaned dataset successfully saved to: '{output_path}'")
print("Final Dataset Shape:", df.shape)

Number of duplicate rows found: 0
=== Step 4 Complete ===
Cleaned dataset successfully saved to: 'dataset/cleaned_ecommerce_sales_150k.csv'
Final Dataset Shape: (138116, 47)


### Master Cleaning Script
* **Problem:** Want a single code block that runs all cleaning steps sequentially from top to bottom[cite: 3, 7].
* **What I Did:** Combined financial recalculations, missing value fills, text standardization, and date parsing into one block and re-saved the clean CSV file[cite: 3, 7].

In [9]:
import pandas as pd

# 1. Load raw dataset
df = pd.read_csv("dataset/ecommerce_sales_customer_analytics_150k.csv")

# 2. Recalculate financial fields
df['net_sales'] = (df['gross_sales'] - df['discount_amount']).round(2)
df['profit'] = (df['net_sales'] - df['product_cost']).round(2)
df['profit_margin_percentage'] = ((df['profit'] / df['net_sales']) * 100).round(2)

# 3. Handle missing values
df['return_status'] = df['return_status'].fillna('Not Returned')
df['return_reason'] = df['return_reason'].fillna('None')
df['campaign_name'] = df['campaign_name'].fillna('Organic/Direct')
df['coupon_code'] = df['coupon_code'].fillna('No Coupon')
df['review_sentiment'] = df['review_sentiment'].fillna('No Feedback')
df['customer_review'] = df['customer_review'].fillna('No Review')
df['delivery_days'] = df['delivery_days'].fillna(df['delivery_days'].median())
df['estimated_delivery_days'] = df['estimated_delivery_days'].fillna(df['estimated_delivery_days'].median())
df['customer_rating'] = df['customer_rating'].fillna(df['customer_rating'].median())

# 4. Standardize text and dates
df['order_date'] = pd.to_datetime(df['order_date'])
text_cols = ['sales_channel', 'order_status', 'payment_method', 'shipping_method', 'customer_segment', 'gender']
for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

# 5. Overwrite the cleaned CSV file
df.to_csv("dataset/cleaned_ecommerce_sales_150k.csv", index=False)
print("Updated cleaned_ecommerce_sales_150k.csv successfully!")

Updated cleaned_ecommerce_sales_150k.csv successfully!


### Verifying Financial Math (Before vs After)
* **Problem:** Need to double-check that our financial math fixes actually saved correctly in the cleaned CSV[cite: 3, 7].
* **What I Did:** Loaded raw and cleaned files side-by-side to compare `gross_sales`, `discount_amount`, `net_sales`, and `profit` on the first 5 rows[cite: 3, 7].

In [10]:
import pandas as pd

# Load both datasets
df_raw = pd.read_csv("dataset/ecommerce_sales_customer_analytics_150k.csv")
df_clean = pd.read_csv("dataset/cleaned_ecommerce_sales_150k.csv")

# Select financial columns for comparison on first 5 rows
fin_cols = ['gross_sales', 'discount_amount', 'net_sales', 'product_cost', 'profit']

print("=== ORIGINAL FINANCIALS (Has Math Miscalculations) ===")
print(df_raw[fin_cols].head())

print("\n=== CLEANED FINANCIALS (Fixed Math) ===")
print(df_clean[fin_cols].head())

=== ORIGINAL FINANCIALS (Has Math Miscalculations) ===
   gross_sales  discount_amount  net_sales  product_cost   profit
0      1350.19           477.89     945.40        588.33   345.04
1      3144.64          1456.06    2018.41       2031.88   -22.47
2       522.81            97.30     468.35        257.73   197.57
3       544.62            53.34     546.52        277.84   247.83
4      2474.52           133.63    2785.27       1231.88  1530.36

=== CLEANED FINANCIALS (Fixed Math) ===
   gross_sales  discount_amount  net_sales  product_cost   profit
0      1350.19           477.89     872.30        588.33   283.97
1      3144.64          1456.06    1688.58       2031.88  -343.30
2       522.81            97.30     425.51        257.73   167.78
3       544.62            53.34     491.28        277.84   213.44
4      2474.52           133.63    2340.89       1231.88  1109.01


### Verifying Missing Values (Before vs After)
* **Problem:** Confirm that null values were replaced as intended[cite: 3, 7].
* **What I Did:** Compared missing value counts between raw and cleaned files, which showed `return_reason` was accidentally left unfilled in that specific variable run[cite: 3, 7].

In [11]:
# Select columns that originally had missing values
missing_cols = ['return_status', 'return_reason', 'campaign_name', 'coupon_code', 'customer_rating']

print("=== ORIGINAL MISSING VALUES COUNT ===")
print(df_raw[missing_cols].isnull().sum())

print("\n=== CLEANED MISSING VALUES COUNT ===")
print(df_clean[missing_cols].isnull().sum())

print("\n=== SAMPLE CLEANED LABELS ===")
print(df_clean[missing_cols].head())

=== ORIGINAL MISSING VALUES COUNT ===
return_status      128654
return_reason      128654
campaign_name       83233
coupon_code        110502
customer_rating     24557
dtype: int64

=== CLEANED MISSING VALUES COUNT ===
return_status           0
return_reason      128654
campaign_name           0
coupon_code             0
customer_rating         0
dtype: int64

=== SAMPLE CLEANED LABELS ===
  return_status return_reason     campaign_name coupon_code  customer_rating
0  Not Returned           NaN  Default_Campaign   No Coupon              3.5
1  Not Returned           NaN  Default_Campaign   No Coupon              3.6
2  Not Returned           NaN    Organic/Direct   No Coupon              4.3
3  Not Returned           NaN    Organic/Direct   No Coupon              3.5
4  Not Returned           NaN    Organic/Direct   No Coupon              3.0


### Fixing Missing Values in `return_reason`
* **Problem:** `return_reason` still had missing values in `df_clean`[cite: 3, 7].
* **What I Did:** Applied `.fillna('None')` specifically to `return_reason`, saved the CSV again, and confirmed missing counts dropped to 0 across all columns[cite: 3, 7].

In [12]:
# 1. Fill return_reason missing values
df_clean['return_reason'] = df_clean['return_reason'].fillna('None')

# 2. Overwrite the cleaned CSV file
df_clean.to_csv("dataset/cleaned_ecommerce_sales_150k.csv", index=False)

# 3. Re-check missing counts across all columns
print("=== UPDATED CLEANED MISSING VALUES COUNT ===")
print(df_clean[missing_cols].isnull().sum())

print("\n=== UPDATED SAMPLE LABELS ===")
print(df_clean[missing_cols].head())

=== UPDATED CLEANED MISSING VALUES COUNT ===
return_status      0
return_reason      0
campaign_name      0
coupon_code        0
customer_rating    0
dtype: int64

=== UPDATED SAMPLE LABELS ===
  return_status return_reason     campaign_name coupon_code  customer_rating
0  Not Returned          None  Default_Campaign   No Coupon              3.5
1  Not Returned          None  Default_Campaign   No Coupon              3.6
2  Not Returned          None    Organic/Direct   No Coupon              4.3
3  Not Returned          None    Organic/Direct   No Coupon              3.5
4  Not Returned          None    Organic/Direct   No Coupon              3.0


### Checking Dataset Dimensions & Data Types
* **Problem:** Ensure no rows were deleted accidentally and check `order_date` data type[cite: 3, 7].
* **What I Did:** Compared row/column shapes between both datasets and verified 0 nulls remain[cite: 3, 7].

In [13]:
# Compare data types between raw and cleaned files
print("=== ORIGINAL DATA TYPES ===")
print("order_date type:", df_raw['order_date'].dtype)

print("\n=== CLEANED DATA TYPES ===")
print("order_date type:", df_clean['order_date'].dtype)

print("\n=== TOTAL DATASET SHAPE CHECK ===")
print("Original Shape:", df_raw.shape)
print("Cleaned Shape :", df_clean.shape)
print("Total Nulls Remaining in Clean File:", df_clean.isnull().sum().sum())

=== ORIGINAL DATA TYPES ===
order_date type: object

=== CLEANED DATA TYPES ===
order_date type: object

=== TOTAL DATASET SHAPE CHECK ===
Original Shape: (138116, 46)
Cleaned Shape : (138116, 46)
Total Nulls Remaining in Clean File: 0


### Converting Date Column to Datetime
* **Problem:** Reading a CSV reloads dates as plain strings (`object` type)[cite: 3, 7].
* **What I Did:** Converted `df_clean['order_date']` to `datetime64[ns]` in memory so it's ready for Week 2 time-series charts[cite: 3, 7].

In [14]:
# Convert order_date to proper datetime format for analysis
df_clean['order_date'] = pd.to_datetime(df_clean['order_date'])

# Verify the updated data type
print("order_date data type:", df_clean['order_date'].dtype)

order_date data type: datetime64[ns]


### Verifying Text Consistency
* **Problem:** Double-check that text values look clean and standardized across categories[cite: 3, 7].
* **What I Did:** Printed unique values for `sales_channel`, `order_status`, `payment_method`, `shipping_method`, `customer_segment`, and `gender` to confirm proper formatting[cite: 3, 7].

In [15]:
# List key categorical text columns
text_cols = ['sales_channel', 'order_status', 'payment_method', 'shipping_method', 'customer_segment', 'gender']

print("=== ORIGINAL CATEGORICAL VALUES (RAW) ===")
for col in text_cols:
    print(f"{col}: {df_raw[col].unique()[:4]}")  # Shows first 4 unique values

print("\n=== CLEANED CATEGORICAL VALUES (CLEANED) ===")
for col in text_cols:
    print(f"{col}: {df_clean[col].unique()[:4]}")

=== ORIGINAL CATEGORICAL VALUES (RAW) ===
sales_channel: ['Mobile App' 'Website' 'Social Media' 'Marketplace']
order_status: ['Completed' 'Pending' 'Cancelled' 'Returned']
payment_method: ['Digital Wallet' 'Debit Card' 'Cash on Delivery' 'Credit Card']
shipping_method: ['Standard' 'Economy' 'Express' 'Same Day']
customer_segment: ['Consumer' 'Premium' 'VIP' 'Business']
gender: ['Male' 'Female' 'Non-Binary']

=== CLEANED CATEGORICAL VALUES (CLEANED) ===
sales_channel: ['Mobile App' 'Website' 'Social Media' 'Marketplace']
order_status: ['Completed' 'Pending' 'Cancelled' 'Returned']
payment_method: ['Digital Wallet' 'Debit Card' 'Cash On Delivery' 'Credit Card']
shipping_method: ['Standard' 'Economy' 'Express' 'Same Day']
customer_segment: ['Consumer' 'Premium' 'Vip' 'Business']
gender: ['Male' 'Female' 'Non-Binary']
